# 第 6 章 · eager 与 graph：`torch.compile` 里面的三道工序

本章回答：默认的"写一行跑一行"（eager）有什么代价？`torch.compile` 加一行就变快，它在背后做了哪几步？什么是"断图"（graph break）和"重编译"（recompile）？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/06-compile-stack.md`（讲解、动画、练习都在讲义里）。对应原文：[00-overview.md](../00-overview.md) §2.5；AISPE ch14 §14.1「PyTorch 编译器深入剖析」（`ai-infra/materials/ai-systems-performance-engineering/docs/chapters/ch14.md` 第 19–673 行：14.1.1 Dynamo 第 40 行、14.1.2 AOT Autograd 第 209 行、14.1.4 Inductor 第 298 行）；官方 torch.compile Programming Model（〔在线〕）；本章输出都在本机 torch 2.15.0 CPU 上跑过（代码见本章末尾"代码对照"）

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import re

from torch._inductor.utils import run_and_get_code
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 融合 | 四个算子一个 kernel | `fused_kernel_names(f, x)` | `cpp_fused_cos_mul_sin_sum_0` |
| 第三步 不断图 | — | `graph_breaks(f, x)` | (1, 0) |
| 第三步 断图 | 依赖张量值的 if | `graph_breaks(g, x)` | (2, 1) |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `f`**（来自 `ch06_compile_stack.py`）：下面要用到的小工具。

In [ ]:
def f(x):
    return (x.sin().cos() * 2).sum(-1)

**函数 `fused_kernel_names`**（来自 `ch06_compile_stack.py`）：编译 fn，返回生成代码里的融合 kernel 名字。

In [ ]:
def fused_kernel_names(fn, *args) -> set:
    """编译 fn，返回生成代码里的融合 kernel 名字。"""
    torch._dynamo.reset()
    _, code = run_and_get_code(torch.compile(fn), *args)
    return set(re.findall(r"cpp_fused\w*", code[0]))

**函数 `g`**（来自 `ch06_compile_stack.py`）：下面要用到的小工具。

In [ ]:
def g(x):
    y = x.sin()
    if y.sum() > 0:
        y = y * 2
    return y.cos()

**函数 `graph_breaks`**（来自 `ch06_compile_stack.py`）：下面要用到的小工具。

In [ ]:
def graph_breaks(fn, *args):
    e = torch._dynamo.explain(fn)(*args)
    return e.graph_count, e.graph_break_count

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
x = torch.randn(64, 64)

In [ ]:
print("\n第二步 四个算子融成：", fused_kernel_names(f, x))

In [ ]:
print("\n第三步 断图：f 的 (图数, 断图数) =", graph_breaks(f, x), "；g =", graph_breaks(g, x))

In [ ]:
print("\n第四步 形状变化：TORCH_LOGS=recompiles 运行可看到只重编译一次")

## 3. 代码里才看得到的

1. **先 `torch._dynamo.reset()`**：同一进程里编译过的函数会被缓存，不 reset 就拿不到新生成的代码。
2. **CPU 上第一次编译要几秒**：Inductor 要调本机 C++ 编译器。

## 4. 自己改一改

- 把 `g` 里的 `if` 改成 `torch.where(y.sum() > 0, y * 2, y)`，再看断图数。
- 用 `TORCH_LOGS=recompiles uv run python ...` 依次喂长度 4、8、16 的输入，数重编译几次。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> eager 一步一单一趟显存，compile 三道工序：Dynamo 抄图，AOTAutograd 配反向，Inductor 融合写代码；
> 看张量的值走分支就断图，用 where 改；guards 不过就重编，形状在变编成动态。